# Label Studio JSON → CSV

Converts Label Studio annotation JSON files into a flat CSV with one row per article.

**Columns produced:**
- Article metadata: `ORN`, `TI`, `YR`, `AU`, `JN`, `AB`, `MD`, `Methodology_Category`, `Period` (where available)
- Annotation counts: `n_variables`, `n_hierarchy`, `n_directional`, `n_correlational`, `n_moderation`
- Edge strings (semicolon-delimited): `variables`, `hierarchy`, `directional`, `correlational`, `moderation`
- Annotation provenance: `source_file`, `task_id`, `annotator_id`


In [ ]:
import json
import re
import os
import pandas as pd
from collections import Counter
print('Imports OK')


In [ ]:
# ── Configure input files and output path ────────────────────────────────────

INPUT_FILES = [
 "<path_to_data>",
 "<path_to_data>",
 "<path_to_data>"

]

# project-13 and project-16 were exported with only AB (no title, ORN, year, etc.).
# We enrich them by matching on abstract text (AB) against this full-metadata file,
# which contains all 74 articles including all 28 from project-13 + project-16.
METADATA_SOURCE = "<path_to_data>"

# Sparse files that need metadata enrichment (matched by AB text)
SPARSE_FILES = {
   "project-13-at-2026-04-19-15-49-d9956b2c.json",
   "project-16-at-2026-04-19-15-50-95a64458.json",
   "project-19-at-2026-04-19-15-49-5cf9db85.json"

}

OUTPUT_DIR = '../data/raw/annotation_csv'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Input files:', [f.split('/')[-1] for f in INPUT_FILES])
print('Metadata source:', METADATA_SOURCE.split('/')[-1])
print('Output dir :', OUTPUT_DIR)


In [ ]:
# ── Build AB → metadata lookup from the full-metadata file ──────────────────
# Used to enrich project-13 and project-16 which only have the AB field.

with open(METADATA_SOURCE) as f:
    meta_data = json.load(f)

# Key: first 200 chars of AB (same as used for matching elsewhere in this project)
ab_to_meta = {}
for item in meta_data:
    ab = item.get('data', {}).get('AB', '').strip()[:200]
    if ab:
        ab_to_meta[ab] = {
            'data': item['data'],
            'id': item.get('id', '')  # 保存task_id用于覆盖
        }

print(f'Metadata lookup built: {len(ab_to_meta)} abstracts from {METADATA_SOURCE.split("/")[-1]}')

# Verify all sparse-file abstracts are covered
for path in INPUT_FILES:
    fname = os.path.basename(path)
    if fname not in SPARSE_FILES:
        continue
    with open(path) as f:
        data = json.load(f)
    missing = [item for item in data
               if item.get('data', {}).get('AB', '').strip()[:200] not in ab_to_meta]
    if missing:
        print(f'WARNING: {fname} — {len(missing)} abstracts NOT found in metadata source:')
        for m in missing:
            print(f'  {m["data"].get("AB","")[:60]!r}')
    else:
        with open(path) as f:
            n = len(json.load(f))
        print(f'{fname}: all {n} abstracts matched in metadata source ✓')

In [ ]:
# ── Parser ───────────────────────────────────────────────────────────────────

def _edge_type(label: str) -> str:
    if label == 'hierarchy':            return 'hierarchy'
    if label.startswith('direction'):   return 'directional'
    if label.startswith('correlation'): return 'correlational'
    if label.startswith('moderation'):  return 'moderation'
    return None


def _validation(label: str) -> str:
    """Extract validation state from a relation label, e.g. direction__validated → validated."""
    if '__' in label:
        return label.split('__', 1)[1]
    return ''   # hierarchy has no validation suffix


def parse_item(item: dict, source_file: str,
               ab_to_meta: dict = None) -> dict:
    """
    Parse one Label Studio task into a flat dict.
    For sparse files (only AB field), metadata is looked up from ab_to_meta
    using the abstract text as key.
    """
    data = item.get('data', {})
    task_id = item.get('id', '')

    # For sparse files, enrich with metadata from the lookup table
    if ab_to_meta is not None:
        ab_key = data.get('AB', '').strip()[:200]
        enriched_item = ab_to_meta.get(ab_key, {})
        if enriched_item:
            # 用metadata source的task_id覆盖原文件的task_id
            task_id = enriched_item.get('id', task_id)
            enriched = enriched_item.get('data', {})
            # Merge: file's own fields take priority; lookup fills missing ones
            data = {**enriched, **data}

    # ── Metadata ──────────────────────────────────────────────────────────────
    row = {
        'source_file':          source_file,
        'task_id':              task_id,
        'ORN':                  data.get('ORN', ''),
        'TI':                   data.get('TI',  ''),
        'YR':                   data.get('YR',  ''),
        'AU':                   data.get('AU',  ''),
        'JN':                   data.get('JN',  ''),
        'AB':                   data.get('AB',  '').strip(),
        'MD':                   data.get('MD',  ''),
        'Methodology_Category': data.get('Methodology_Category', ''),
        'Period':               data.get('Period', ''),
    }

    # ── Annotation ────────────────────────────────────────────────────────────
    anns = item.get('annotations', [])
    if not anns:
        row.update({
            'annotator_id': '', 'n_variables': 0,
            'variables': '', 'n_hierarchy': 0, 'hierarchy': '',
            'n_directional': 0, 'directional': '',
            'n_correlational': 0, 'correlational': '',
            'n_moderation': 0, 'moderation': '',
        })
        return row

    ann = anns[0]
    row['annotator_id'] = ann.get('completed_by', '')
    result = ann.get('result', [])

    # Build id → variable text map
    vm = {}
    for r in result:
        if r.get('type') == 'labels' and 'Variable' in r['value'].get('labels', []):
            vm[r['id']] = r['value'].get('text', '').strip()
    # Apply textarea renames
    for r in result:
        if r.get('type') == 'textarea' and r.get('from_name') == 'var_name' and r['id'] in vm:
            nn = r['value'].get('text', '')
            if isinstance(nn, list): nn = nn[0] if nn else ''
            if nn.strip(): vm[r['id']] = nn.strip()

    # Parse edges — loop over ALL labels per relation (multi-label support)
    hierarchy_edges     = []
    directional_edges   = []
    correlational_edges = []
    moderation_raw      = []   # (moderator_text, target_text, validation)

    for r in result:
        if r.get('type') != 'relation': continue
        fid, tid = r.get('from_id', ''), r.get('to_id', '')
        if fid not in vm or tid not in vm: continue
        src, tgt = vm[fid], vm[tid]
        for lbl in r.get('labels', []):
            et  = _edge_type(lbl)
            val = _validation(lbl)
            if et == 'hierarchy':
                hierarchy_edges.append(f'{src} -> {tgt}')
            elif et == 'directional':
                directional_edges.append(f'{src} -> {tgt} [{val}]')
            elif et == 'correlational':
                correlational_edges.append(f'{src} <-> {tgt} [{val}]')
            elif et == 'moderation':
                moderation_raw.append((src, tgt, val))

    # Group moderation: (moderator, validation) → [moderated_variables]
    mod_groups = {}
    for mod, tgt, val in moderation_raw:
        mod_groups.setdefault((mod, val), []).append(tgt)
    moderation_edges = [
        f"{mod} moderates ({', '.join(targets)}) [{val}]"
        for (mod, val), targets in mod_groups.items()
    ]

    row.update({
        'n_variables':    len(vm),
        'variables':      '; '.join(sorted(vm.values())),
        'n_hierarchy':    len(hierarchy_edges),
        'hierarchy':      '; '.join(hierarchy_edges),
        'n_directional':  len(directional_edges),
        'directional':    '; '.join(directional_edges),
        'n_correlational': len(correlational_edges),
        'correlational':  '; '.join(correlational_edges),
        'n_moderation':   len(moderation_edges),
        'moderation':     '; '.join(moderation_edges),
    })
    return row


print('parse_item() defined.')

In [ ]:
# ── Convert each file ────────────────────────────────────────────────────────

all_rows  = []   # combined across all files
per_file_dfs = {}

for path in INPUT_FILES:
    fname = os.path.basename(path)
    with open(path) as f:
        data = json.load(f)

    # Pass metadata lookup for sparse files (project-13, project-16)
    meta_lookup = ab_to_meta if fname in SPARSE_FILES else None
    rows = [parse_item(item, fname, meta_lookup) for item in data]
    df   = pd.DataFrame(rows)

    # Save per-file CSV
    out_name = fname.replace('.json', '.csv')
    out_path = os.path.join(OUTPUT_DIR, out_name)
    df.to_csv(out_path, index=False)

    per_file_dfs[fname] = df
    all_rows.extend(rows)

    # Summary
    et = Counter()
    et['hierarchy']     = df['n_hierarchy'].sum()
    et['directional']   = df['n_directional'].sum()
    et['correlational'] = df['n_correlational'].sum()
    et['moderation']    = df['n_moderation'].sum()
    print(f'{fname}: {len(df)} articles → {out_path}')
    print(f'  vars={df["n_variables"].sum()}  '
          f'hier={et["hierarchy"]}  '
          f'dir={et["directional"]}  '
          f'corr={et["correlational"]}  '
          f'mod={et["moderation"]}')

# Save combined CSV
df_all = pd.DataFrame(all_rows)
combined_path = os.path.join(OUTPUT_DIR, 'coder_B_r.csv')
df_all.to_csv(combined_path, index=False)

print(f'\nCombined: {len(df_all)} articles → {combined_path}')

In [ ]:
# ── Preview ───────────────────────────────────────────────────────────────────

print(f'Combined CSV: {len(df_all)} rows × {len(df_all.columns)} columns')
print(f'Columns: {df_all.columns.tolist()}')
print()
display(df_all[[
    'source_file', 'task_id', 'ORN', 'TI', 'YR', 'JN',
    'n_variables', 'n_hierarchy', 'n_directional', 'n_correlational', 'n_moderation'
]].head(10))
print()

# Edge count summary by source file
print('Edge count summary by source file:')
summary = df_all.groupby('source_file')[[
    'n_variables','n_hierarchy','n_directional','n_correlational','n_moderation'
]].agg(['sum','mean']).round(2)
display(summary)
